# 合并 LoRA Adapter → 完整 bfloat16 权重（Notebook 版）

训练完得到 LoRA adapter（~250 MB）后，把它合并回基础模型，得到一个独立的 bf16 完整权重（~65 GB），便于后续推理直接 `from_pretrained` 加载。

**前置**：训练已完成、`adapter/` 目录已存在。

In [ ]:
# ============================================================
# 设置 HF 环境变量 + 加载 yaml 配置
# ============================================================
import os
from pathlib import Path
import yaml

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'src' / 'Fine_tuning' / 'configs' / 'train_config.yaml').exists():
    if REPO_ROOT.parent == REPO_ROOT:
        raise RuntimeError('找不到 train_config.yaml')
    REPO_ROOT = REPO_ROOT.parent

with open(REPO_ROOT / 'src' / 'Fine_tuning' / 'configs' / 'train_config.yaml', 'r', encoding='utf-8') as f:
    CFG = yaml.safe_load(f)

for k, v in (CFG.get('env') or {}).items():
    if v is not None:
        os.environ.setdefault(k, str(v))

print('HF_HOME =', os.environ.get('HF_HOME'))

In [ ]:
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

base_model_name = CFG['model']['name_or_path']
adapter_dir = REPO_ROOT / CFG['adapter_dir']
merged_dir  = REPO_ROOT / CFG['merged_dir']
merged_dir.mkdir(parents=True, exist_ok=True)

print('[路径] base_model :', base_model_name)
print('[路径] adapter_dir:', adapter_dir)
print('[路径] merged_dir :', merged_dir)
assert adapter_dir.exists(), f'adapter 目录不存在：{adapter_dir}'

In [ ]:
# ============================================================
# 加载基础模型（bf16 完整精度，不量化；A100 80GB 可装下）
# ============================================================
DTYPE = torch.bfloat16

base = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=DTYPE,                    # transformers 4.45+ 用 dtype 替代 torch_dtype
    device_map='auto',
    trust_remote_code=CFG['model'].get('trust_remote_code', True),
)
print('[加载] base model OK')

In [ ]:
# ============================================================
# 挂上 LoRA adapter 并 merge_and_unload
# ============================================================
peft_model = PeftModel.from_pretrained(base, str(adapter_dir), dtype=DTYPE)
print('[加载] LoRA adapter OK')

merged_model = peft_model.merge_and_unload()
print('[合并] merge_and_unload OK')

In [ ]:
# ============================================================
# 保存合并后的完整权重 + tokenizer
# ============================================================
merged_model.save_pretrained(str(merged_dir), safe_serialization=True)

tokenizer = AutoTokenizer.from_pretrained(
    str(adapter_dir),
    trust_remote_code=CFG['model'].get('trust_remote_code', True),
)
tokenizer.save_pretrained(str(merged_dir))

print('[完成] 合并后完整权重 →', merged_dir)
print('       后续推理：AutoModelForCausalLM.from_pretrained(<merged_dir>, trust_remote_code=True)')